In [1]:
import pyarrow.dataset as ds

dataset = ds.dataset(
    "../output/illustris_skirt.parquet",
    format="parquet",
    exclude_invalid_files=True,
)
print(dataset.count_rows())
dataset.schema

376


image: list<element: list<element: list<element: float>>>
  child 0, element: list<element: list<element: float>>
      child 0, element: list<element: float>
          child 0, element: float
simulation: string
snapshot: int32
subhalo_id: int32
-- schema metadata --
huggingface: '{"info": {"features": {"image": {"feature": {"feature": {"f' + 259

In [2]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyarrow.compute as pc

n_cols = 5
n_rows = 4
page_size = n_cols * n_rows

search_box = widgets.Text(value="", placeholder="Enter subhalo_id", description="Subhalo ID:", continuous_update=False)
slider = widgets.IntSlider(value=0, min=0, max=0, description="Page:")
out = widgets.Output()

_cached_table = None


def get_filtered_table():
    global _cached_table
    sid_text = search_box.value.strip()
    columns = ["image", "simulation", "snapshot", "subhalo_id"]
    if sid_text:
        try:
            filt = pc.equal(ds.field("subhalo_id"), int(sid_text))
            _cached_table = dataset.to_table(columns=columns, filter=filt)
        except ValueError:
            _cached_table = dataset.to_table(columns=columns)
    else:
        _cached_table = dataset.to_table(columns=columns)
    return _cached_table


def update_slider(*args):
    table = get_filtered_table()
    n_pages = max(1, (len(table) + page_size - 1) // page_size)
    slider.max = n_pages - 1
    slider.value = 0
    show_page(0)


def show_page(page):
    if _cached_table is None:
        return
    table = _cached_table.slice(page * page_size, page_size).to_pydict()
    batch = table["image"]
    simulations = table["simulation"]
    snapshots = table["snapshot"]
    subhalo_ids = table["subhalo_id"]
    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, 20))
        for ax, img_channels, sim, snap, sid in zip(axes.flatten(), batch, simulations, snapshots, subhalo_ids):
            data = np.stack([np.stack(ch) for ch in img_channels]).transpose(1, 2, 0) * 255
            image = Image.fromarray(data.astype(np.uint8), "RGB")
            ax.imshow(image)
            ax.set_title(f"{sim} / {snap} / {sid}", fontsize=14)
            ax.axis("off")
        for ax in axes.flatten()[len(batch) :]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)


search_box.observe(update_slider, names="value")
slider.observe(lambda change: show_page(change["new"]), names="value")
display(search_box, slider, out)
update_slider()


Text(value='', continuous_update=False, description='Subhalo ID:', placeholder='Enter subhalo_id')

IntSlider(value=0, description='Page:', max=0)

Output()